#**Etapa Silver**

Como primer paso de esta etapa, vamos a echar un vistazo general a las tablas generadas en bronce.

El proceso de visualización y limpieza se detalla para la primer tabla a analizar, como es un proceso reiterativo, para tablas posteriores solo se detallará cambios y decisiones particulares no desarrolladas anteriormente.

Se realizan las configuraciones previas de rutas, al igual que en la etapa bronce.

A lo largo de la etapa, se agregaran ciertas columnas que se repetirán para cada tabla de nuestro set de datos, se explican a continuación.

Las columnas con sufijo `_tipado`: contienen la conversión de los valores originales y se mantienen temporalmente para comparar resultados.

`errores_conversion` identifica campos informados que no pudieron convertirse.

`problemas_validacion` identifica valores que requieren revisión por reglas de dominio. 

La selección final sustituirá las columnas originales por las convertidas, conservando por separado los casos pendientes.

In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema_bronze", "movilidad_bronze")
dbutils.widgets.text("schema_silver", "movilidad_silver")

CATALOG = dbutils.widgets.get("catalog")
BRONZE = dbutils.widgets.get("schema_bronze")
SILVER = dbutils.widgets.get("schema_silver")

#crea el schema si  ya no  existe
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER}")

print("Tablas de origen:", f"{CATALOG}.{BRONZE}")
print("Tablas de destino:", f"{CATALOG}.{SILVER}")

## **Viajes_STM**

Nuestra primer tabla corresponde a los viajes realizados, que contiene las siguientes columnas:

`id_viaje`:	Identificador del viaje dentro del mes. Un viaje reúne tramos realizados con un único pago y puede involucrar varios pasajeros.

`con_tarjeta`:	Indica si el viaje se realiza con tarjeta: 1 sí, 0 no.

`fecha_evento`	Fecha y hora en que se registra el tramo.

`tipo_viaje`:	Código de la categoría de viaje, considerando características del viaje y del usuario.

`descripcion_tipo_viaje`:	Descripción de esa categoría.

`grupo_usuario`:	Código del grupo de usuario.

`descripcion_grupo_usuario`:	Descripción del grupo, por ejemplo estudiante o jubilado.

`grupo_usuario_especifico`:	Código del subgrupo dentro del grupo de usuario.

`descripcion_grupo_usuario_espe`:	Descripción del subgrupo.

`ordinal_de_tramo`:	Posición del tramo dentro del viaje; el diccionario lo define para viajes con tarjeta.

`cantidad_pasajeros`:	Cantidad de personas que realizan el tramo.

`codigo_parada_origen`	: Código de la parada donde se produce el ascenso.

`cod_empresa`:	Código de la empresa transportista.

`descrip_empresa`:	Nombre o descripción de la empresa.

`linea_codigo`:	Código interno de la línea.

`dsc_linea`:	Nombre público de la línea.

`sevar_codigo`:	Código de la variante del recorrido.

`archivo_origen`:	Código del recurso mensual conservado por la ingesta.

`periodo_fuente`.	Período del recurso normalizado como AAAAMM.


In [ ]:
viajes_bronze = spark.table(
    f"{CATALOG}.{BRONZE}.viajes_stm"
)
#como está dando problemas para leerla desde bronce, se vuelve a construir desde acá
if "periodo_fuente" not in viajes_bronze.columns:

    if "archivo_origen" not in viajes_bronze.columns:
        raise ValueError(
            "No existen periodo_fuente ni archivo_origen "
            "para recuperar el período."
        )

    codigo = F.trim(F.col("archivo_origen"))

    viajes_bronze = viajes_bronze.withColumn(
        "periodo_fuente",
        F.when(
            codigo.rlike(r"^20[0-9]{2}(0[1-9]|1[0-2])$"),
            codigo
        )
        .when(
            codigo.rlike(r"^(0[1-9]|1[0-2])20[0-9]{2}$"),
            F.concat(
                F.substring(codigo, 3, 4),
                F.substring(codigo, 1, 2)
            )
        )
        .otherwise(F.lit(None).cast("string"))
    )

viajes_bronze.printSchema()
display(viajes_bronze.limit(10))

**Mostrar los registros que tengan nulos, vacíos o espacios**

Se detectan aquellos registros con valores nulos imprimiendo para estas el resto de columnas, para así decidir qué hacer con ellas evaluando el panorame completo.

La columna `columnas_faltantes` indica qué campos revisar en cada registro, la visualización puede mostrar solo una parte si hay muchos resultados.

In [ ]:
columnas_originales = viajes_bronze.columns

condiciones_faltantes = [
    F.when(
        F.col(c).isNull()
        | (F.trim(F.col(c).cast("string")) == ""),
        F.lit(c)
    )
    for c in columnas_originales
]

viajes_revisados = viajes_bronze.withColumn(
    "columnas_faltantes",
    F.filter(
        F.array(*condiciones_faltantes),
        lambda elemento: elemento.isNotNull()
    )
)

registros_con_faltantes = viajes_revisados.filter(
    F.size("columnas_faltantes") > 0
)

# Visualización opcional: puede mostrar muchos registros
# display(registros_con_faltantes)

#descomentar si se quiere evaluar una columna concreta
"""
display(
    registros_con_faltantes.filter(
        F.array_contains(
            F.col("columnas_faltantes"),
            "ordinal_de_tramo"
        )
    )
)
"""

**Cantidad de faltantes por columna**

Resumen de nulos sobre toda la tabla

In [ ]:
resumen_faltantes = viajes_revisados.agg(
    F.count("*").alias("total_registros"),

    *[
        F.count(
            F.when(
                F.array_contains(
                    F.col("columnas_faltantes"), c
                ),
                F.lit(1)
            )
        ).alias(c)
        for c in columnas_originales
    ]
)

display(resumen_faltantes)

**Decisiones para cada columna con datos faltantes**

Para cada caso, se cerciora que la correspondencia entre tipo y descripcion sea unico, es decir, que un tipo no tenga 2 posibles descripciones,

`tipo_viaje`: Completar desde una descripción solo si identifica un código único en un contexto temporal compatible. Si también falta la descripción, se conserva nulo.

`descripcion_tipo_viaje`: Completar desde tipo_viaje mediante una referencia.

`descripcion_grupo_usuario`: Completar desde grupo_usuario.

`descripcion_grupo_usuario_espe`: Completar desde la combinación grupo_usuario + grupo_usuario_especifico.

`cod_empresa`: Completar desde una descripción de empresa inequívoca o una referencia oficial compatible. Si ambos faltan, se manteniene nulo.

`descrip_empresa`: Completar desde cod_empresa.

`linea_codigo`: Completar desde empresa + nombre público de línea.

`dsc_linea`: Completar desde el código de línea y, cuando corresponda, empresa.

`sevar_codigo`: No imputar automáticamente: necesita evidencia suficiente para determinar una única variante.


Los **string vacios se cambian a null**, los **registros con esapcio se le aplica trim**, limpiandolos.

In [ ]:
viajes_limpios = viajes_bronze.select(
    *[
        F.when(
            F.trim(F.col(c)) == "",
            F.lit(None).cast("string")
        )
        .otherwise(F.trim(F.col(c)))
        .alias(c)
        for c in columnas_originales
    ]
)

**Conversión de tipos de datos**

Desde bronce, todos los datos vienen como string extraidos desde el CSV, en esta etapa se convierte cada tipo numérico a `INT`.

Las fechas tambien se estandarizan, teniendo en cuenta los varios formatos presentes: fecha y hora separadas por un espacio y formato ISO con zona horaria.

ejemplo formato fecha + hora: `2025-10-04 13:41:03` 

ejemplo formato ISO: `2021-07-27T15:58:58.000-03:00`

La conversión contempla ambos formatos para evitar que fechas válidas se conviertan en nulos.


In [ ]:
spark.conf.set(
    "spark.sql.session.timeZone",
    "America/Montevideo"
)

tipos_numericos = {
    "id_viaje": "BIGINT", #se hace un cast a BIGINT para que no haya problemas con los ID
    "con_tarjeta": "INT",
    "tipo_viaje": "INT",
    "grupo_usuario": "INT",
    "grupo_usuario_especifico": "INT",
    "ordinal_de_tramo": "INT",
    "cantidad_pasajeros": "INT",
    "codigo_parada_origen": "BIGINT",
    "cod_empresa": "INT",
    "linea_codigo": "INT",
    "sevar_codigo": "INT",
}

viajes_conversion = viajes_limpios

for columna, tipo in tipos_numericos.items():
    viajes_conversion = viajes_conversion.withColumn(
        f"{columna}_tipado",
        F.expr(f"try_cast(`{columna}` AS {tipo})")
    )

viajes_conversion = viajes_conversion.withColumn(
    "fecha_evento_tipado",
    F.coalesce(
        F.try_to_timestamp(
            F.col("fecha_evento"),
            F.lit("yyyy-MM-dd HH:mm:ss")
        ),
        F.try_to_timestamp(
            F.col("fecha_evento"),
            F.lit("yyyy-MM-dd'T'HH:mm:ss.SSSXXX")
        ),
        F.try_to_timestamp(
            F.col("fecha_evento"),
            F.lit("yyyy-MM-dd'T'HH:mm:ssXXX")
        )
    )
)

columnas_convertidas = [
    *tipos_numericos.keys(),
    "fecha_evento"
]

viajes_conversion = viajes_conversion.withColumn(
    "errores_conversion",
    F.filter(
        F.array(*[
            F.when(
                F.col(c).isNotNull()
                & F.col(f"{c}_tipado").isNull(),
                F.lit(c)
            )
            for c in columnas_convertidas
        ]),
        lambda elemento: elemento.isNotNull()
    )
)


# Visualización opcional de errores de conversión
# display(
#     viajes_conversion.filter(
#         F.size("errores_conversion") > 0
#     )
# )

viajes_conversion.printSchema()

### Orden del procesamiento de viajes

El procesamiento sigue este orden:

1. **Normalizar y convertir**: limpiar espacios, convertir cadenas vacías a NULL y convertir los numéricos a INT o BIGINT según el campo y fecha_evento a timestamp.
2. **Eliminar copias exactas**: agrupar registros idénticos en todos sus campos y conservar una única ocurrencia.
3. **Investigar conflictos de tramo**: detectar grupos que compartan la clave candidata (`periodo_fuente`, `id_viaje`, `ordinal_de_tramo`).
4. **Reconstruir faltantes mediante referencias únicas**: completar descripciones y códigos usando correspondencias inequívocas observadas en el mismo período.
5. **Medir resultados**: comparar nulos antes y después del relleno.

La reconstrucción se realiza después de deduplicar para no usar copias como evidencia adicional.

**¿Qué se considera duplicado exacto?**
Un registro es duplicado exacto de otro si todos sus campos de negocio (incluido `periodo_fuente`) son idénticos. La comparación se realiza sobre los registros ya normalizados y convertidos, antes de reconstruir faltantes.

**¿Por qué no se deduplica solo por `id_viaje`?**
Un viaje puede contener varios tramos legítimos con el mismo identificador. Eliminar por `id_viaje` descartaría tramos válidos. La deduplicación se hace sobre el conjunto completo de columnas.

**Diferencia entre copia exacta y conflicto de tramo**
- Una **copia exacta** repite todos los campos; se conserva una sola ocurrencia.
- Un **conflicto de tramo** ocurre cuando dos registros coinciden en la clave candidata (`periodo_fuente`, `id_viaje`, `ordinal_de_tramo`) pero difieren en otros campos. No se eliminan ni se elige arbitrariamente una fila; se señalan para investigación.

In [ ]:
# Comprobación: detener el procesamiento si existen errores de conversión.
# No se deduplican registros problemáticos convertidos a nulos silenciosamente.
# La visualización detallada de errores ya está comentada en la celda anterior.

errores_conversion_count = viajes_conversion.filter(
    F.size("errores_conversion") > 0
).count()

assert errores_conversion_count == 0, (
    f"Hay {errores_conversion_count} registros con errores de conversión. "
    "Revisar la celda de conversión antes de continuar."
)

### Preparar `viajes_base`

A partir de `viajes_conversion`, se seleccionan los campos originales de negocio reemplazando cada campo convertido por su columna `_tipado` y asignándole nuevamente el nombre original.

Se excluyen:
- `archivo_origen`: no forma parte de la identidad del tramo.
- Columnas auxiliares de diagnóstico (`errores_conversion`, `columnas_faltantes`).
- Las versiones originales de los campos que ya fueron convertidos.

Los originales permanecen disponibles en la tabla bronze.

La comparación de duplicados se realiza sobre estos registros normalizados, antes de reconstruir faltantes.

In [ ]:
viajes_base = viajes_conversion.select(
    F.col("id_viaje_tipado").alias("id_viaje"),
    F.col("con_tarjeta_tipado").alias("con_tarjeta"),
    F.col("fecha_evento_tipado").alias("fecha_evento"),
    F.col("tipo_viaje_tipado").alias("tipo_viaje"),
    F.col("descripcion_tipo_viaje"),
    F.col("grupo_usuario_tipado").alias("grupo_usuario"),
    F.col("descripcion_grupo_usuario"),
    F.col("grupo_usuario_especifico_tipado").alias("grupo_usuario_especifico"),
    F.col("descripcion_grupo_usuario_espe"),
    F.col("ordinal_de_tramo_tipado").alias("ordinal_de_tramo"),
    F.col("cantidad_pasajeros_tipado").alias("cantidad_pasajeros"),
    F.col("codigo_parada_origen_tipado").alias("codigo_parada_origen"),
    F.col("cod_empresa_tipado").alias("cod_empresa"),
    F.col("descrip_empresa"),
    F.col("linea_codigo_tipado").alias("linea_codigo"),
    F.col("dsc_linea"),
    F.col("sevar_codigo_tipado").alias("sevar_codigo"),
    F.col("periodo_fuente"),
)

display(viajes_base.limit(10))
viajes_base.printSchema()

### Deduplicación exacta y materialización

Se agrupan los registros de `viajes_base` por todas sus columnas y se cuenta cuántas veces aparece cada combinación. El resultado se materializa como Delta en una tabla intermedia dedicada `{CATALOG}.movilidad_trabajo.viajes_agrupados`.

**¿Por qué se materializa?**
La agrupación sobre el volumen completo de viajes es costosa. Materializar el resultado permite reutilizarlo sin recalcular y leer `viajes_sin_duplicados` directamente desde la tabla guardada. Se usa `overwrite` únicamente sobre esta tabla intermedia; no se modifica bronze ni se publica la tabla silver definitiva.

In [ ]:
# Crear el esquema de trabajo si no existe
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.movilidad_trabajo")

grupos_viajes = (
    viajes_base
    .groupBy(*viajes_base.columns)
    .agg(F.count("*").alias("veces"))
)

# Materializar como Delta en tabla intermedia dedicada.
# No ejecutar count(), display() ni orderBy() antes de la escritura.
(
    grupos_viajes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.movilidad_trabajo.viajes_agrupados")
)

# Leer la tabla materializada
grupos_viajes_guardados = spark.table(
    f"{CATALOG}.movilidad_trabajo.viajes_agrupados"
)

viajes_sin_duplicados = grupos_viajes_guardados.drop("veces")

### Resumen de deduplicación

Se calcula en una única agregación cuántos registros entraron, cuántos quedaron tras eliminar copias exactas y cuántas copias se descartaron. No se fija un resultado esperado: el criterio de comparación cambió respecto de ejecuciones anteriores.

In [ ]:
resumen_deduplicacion = grupos_viajes_guardados.agg(
    F.sum("veces").alias("registros_entrada"),
    F.count("*").alias("registros_sin_duplicados"),
    F.sum(
        F.when(F.col("veces") > 1, F.lit(1)).otherwise(F.lit(0))
    ).alias("grupos_repetidos"),
    F.sum(
        F.when(F.col("veces") > 1, F.col("veces")).otherwise(F.lit(0))
    ).alias("registros_involucrados"),
    F.sum(
        F.when(F.col("veces") > 1, F.col("veces") - 1).otherwise(F.lit(0))
    ).alias("copias_descartadas"),
)

display(resumen_deduplicacion)

# Visualización opcional de ejemplos de grupos repetidos
# display(
#     grupos_viajes_guardados
#     .filter(F.col("veces") > 1)
#     .limit(20)
# )

### Conflictos de tramo

Sobre `viajes_sin_duplicados`, se buscan registros que compartan la clave candidata del tramo: `periodo_fuente`, `id_viaje`, `ordinal_de_tramo`.

Se filtran los registros relevantes:
- `con_tarjeta = 1`
- `periodo_fuente` informado
- `id_viaje` informado
- `ordinal_de_tramo > 0`

Un grupo con más de un registro indica que registros diferentes comparten la clave candidata; no demuestra por sí solo que sean tramos distintos. Estos **conflictos de tramo** no se eliminan ni se resuelven arbitrariamente; se señalan para investigación.

**Nota sobre `ordinal_de_tramo`**: el valor cero en viajes sin tarjeta no debe declararse inválido aplicando automáticamente la regla de viajes con tarjeta. Por eso se analiza por separado la distribución según `con_tarjeta`.

In [ ]:
conflictos_tramos = (
    viajes_sin_duplicados
    .filter(
        (F.col("con_tarjeta") == 1)
        & F.col("periodo_fuente").isNotNull()
        & F.col("id_viaje").isNotNull()
        & (F.col("ordinal_de_tramo") > 0)
    )
    .groupBy(
        "periodo_fuente",
        "id_viaje",
        "ordinal_de_tramo"
    )
    .agg(F.count("*").alias("cantidad"))
    .filter(F.col("cantidad") > 1)
)

display(conflictos_tramos.limit(100))


In [ ]:
# Distribución de registros por con_tarjeta y ordinal_de_tramo
# Permite analizar por separado los casos que no cubre la clave candidata
distribucion_ordinal = (
    viajes_sin_duplicados
    .groupBy("con_tarjeta", "ordinal_de_tramo")
    .agg(F.count("*").alias("registros"))
    .orderBy("con_tarjeta", "ordinal_de_tramo")
)

display(distribucion_ordinal)

### Referencias para reconstrucción de faltantes

Todas las referencias se construyen desde `base_referencias = viajes_sin_duplicados`, es decir, desde los datos ya deduplicados y antes de cualquier relleno. Esto garantiza que no se usen valores imputados como evidencia para construir otras referencias.

**Cómo se garantiza la unicidad:**
Para cada regla, se excluyen filas con objetivo nulo o claves nulas, se agrupa por las claves y se conservan solo los grupos donde `MIN = MAX`. Esto asegura una correspondencia única observada en ese período. No se usan `first()`, valores más frecuentes ni coincidencias aproximadas.

`sevar_codigo` **no se reconstruye** bajo ninguna circunstancia: no hay evidencia suficiente para determinar una variante única. Sus valores existentes y sus nulos se mantienen.

**Algunos nulos permanecerán** cuando no existe referencia para esa combinación de claves, la referencia es ambigua (MIN ≠ MAX) o faltan las claves necesarias para la búsqueda.

In [ ]:
base_referencias = viajes_sin_duplicados

reglas_referencias = [
    {"nombre": "ref_tipo_viaje",          "claves": ["periodo_fuente", "descripcion_tipo_viaje"],        "objetivo": "tipo_viaje"},
    {"nombre": "ref_desc_tipo_viaje",     "claves": ["periodo_fuente", "tipo_viaje"],                    "objetivo": "descripcion_tipo_viaje"},
    {"nombre": "ref_desc_grupo_usuario",  "claves": ["periodo_fuente", "grupo_usuario"],                 "objetivo": "descripcion_grupo_usuario"},
    {"nombre": "ref_desc_grupo_espe",     "claves": ["periodo_fuente", "grupo_usuario", "grupo_usuario_especifico"], "objetivo": "descripcion_grupo_usuario_espe"},
    {"nombre": "ref_cod_empresa",         "claves": ["periodo_fuente", "descrip_empresa"],               "objetivo": "cod_empresa"},
    {"nombre": "ref_descrip_empresa",     "claves": ["periodo_fuente", "cod_empresa"],                  "objetivo": "descrip_empresa"},
    {"nombre": "ref_linea_codigo",        "claves": ["periodo_fuente", "cod_empresa", "dsc_linea"],      "objetivo": "linea_codigo"},
    {"nombre": "ref_dsc_linea",           "claves": ["periodo_fuente", "cod_empresa", "linea_codigo"],   "objetivo": "dsc_linea"},
]

referencias = {}

for regla in reglas_referencias:
    claves = regla["claves"]
    objetivo = regla["objetivo"]
    nombre = regla["nombre"]

    # Excluir filas con objetivo nulo o alguna clave nula
    filtro = F.col(objetivo).isNotNull()
    for c in claves:
        filtro = filtro & F.col(c).isNotNull()

    # Agrupar por claves, conservar solo grupos donde MIN = MAX
    referencias[nombre] = (
        base_referencias
        .filter(filtro)
        .groupBy(*claves)
        .agg(
            F.min(objetivo).alias("valor_min"),
            F.max(objetivo).alias("valor_max"),
        )
        .filter(F.col("valor_min") == F.col("valor_max"))
        .select(*claves, F.col("valor_min").alias("valor_referencia"))
    )

print(f"Se construyeron {len(reglas_referencias)} DataFrames de referencia.")

### Aplicar reconstrucciones

Se parte de `viajes_completados = viajes_sin_duplicados` y se aplican las referencias en el orden A–H mediante `LEFT JOIN` y `COALESCE(valor_actual, valor_referencia)`.

No se reemplazan valores ya informados: el `COALESCE` conserva el valor actual si no es nulo y solo usa la referencia cuando el actual falta.

Las columnas temporales de referencia se eliminan después de cada relleno.

**`sevar_codigo` no se imputa** bajo ninguna circunstancia.

Las claves de una reconstrucción pueden haberse completado en un paso anterior, pero las referencias siempre provienen de `base_referencias`, no de valores ya rellenados.

In [ ]:
viajes_completados = viajes_sin_duplicados

for regla in reglas_referencias:
    nombre = regla["nombre"]
    claves = regla["claves"]
    objetivo = regla["objetivo"]

    viajes_completados = (
        viajes_completados
        .join(referencias[nombre], claves, "left")
        .withColumn(
            objetivo,
            F.coalesce(F.col(objetivo), F.col("valor_referencia"))
        )
        .drop("valor_referencia")
    )

# sevar_codigo no se reconstruye: mantener sus valores existentes y sus nulos
# display(viajes_completados.limit(10))
viajes_completados.printSchema()

### Medir resultados

Se comparan los nulos de las ocho columnas objetivo antes y después del relleno, con una columna `etapa` para distinguir ambas etapas.

Se verifica que el total de filas antes y después del relleno sea igual. Si no coincide, se detiene el proceso: hay una multiplicación o pérdida de registros.

No se comparan los conteos contra los valores iniciales de bronze para atribuir recuperaciones: la deduplicación también puede reducir la cantidad de nulos.

In [ ]:
columnas_objetivo = [
    "tipo_viaje",
    "descripcion_tipo_viaje",
    "descripcion_grupo_usuario",
    "descripcion_grupo_usuario_espe",
    "cod_empresa",
    "descrip_empresa",
    "linea_codigo",
    "dsc_linea",
]

# Resumen antes del relleno: total de filas y nulos por columna objetivo
resumen_antes = viajes_sin_duplicados.agg(
    F.count("*").alias("total_filas"),
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in columnas_objetivo
    ]
).withColumn("etapa", F.lit("antes_relleno"))

# Resumen después del relleno: total de filas y nulos por columna objetivo
resumen_despues = viajes_completados.agg(
    F.count("*").alias("total_filas"),
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in columnas_objetivo
    ]
).withColumn("etapa", F.lit("despues_relleno"))

# Recuperar las dos filas agregadas en una sola acción
filas_resumen = resumen_antes.unionByName(resumen_despues).collect()

# Verificar en Python que el total de filas sea igual
total_antes = filas_resumen[0]["total_filas"]
total_despues = filas_resumen[1]["total_filas"]

assert total_antes == total_despues, (
    f"El total de filas cambió: {total_antes} antes → {total_despues} después. "
    "Posible multiplicación o pérdida de registros."
)

# Mostrar las filas recuperadas sin volver a ejecutar el DataFrame original
display(spark.createDataFrame(filas_resumen))
print(f"Total de filas: {total_antes} (antes) = {total_despues} (después)")

### Validaciones de contenido

Después de la reconstrucción, se verifican condiciones de calidad sobre `viajes_completados` usando los nombres definitivos de las columnas.

Se identifica una columna `problemas_validacion` con motivos específicos. No se eliminan registros automáticamente: varias reglas pueden afectar la misma fila, por lo que los contadores no representan registros distintos.

`sevar_codigo` nulo se conserva como información incompleta, sin imputarlo ni descartar la fila por eso.

No se declaran inválidos todos los códigos cero: existen categorías documentadas como NO CORRESPONDE.

In [ ]:
viajes_completados = viajes_completados.withColumn(
    "problemas_validacion",
    F.filter(
        F.array(
            F.when(
                F.col("con_tarjeta").isNull()
                | ~F.col("con_tarjeta").isin(0, 1),
                F.lit("CON_TARJETA_INVALIDA_O_AUSENTE")
            ),
            F.when(
                F.col("cantidad_pasajeros").isNull()
                | (F.col("cantidad_pasajeros") <= 0),
                F.lit("CANTIDAD_PASAJEROS_INVALIDA")
            ),
            F.when(
                F.col("fecha_evento").isNull(),
                F.lit("FECHA_EVENTO_AUSENTE")
            ),
            F.when(
                F.col("periodo_fuente").isNull()
                | ~F.col("periodo_fuente").rlike(r"^[0-9]{4}(0[1-9]|1[0-2])$"),
                F.lit("PERIODO_FUENTE_INVALIDO")
            ),
            F.when(
                F.col("fecha_evento").isNotNull()
                & F.col("periodo_fuente").rlike(r"^[0-9]{4}(0[1-9]|1[0-2])$")
                & (
                    F.date_format(F.col("fecha_evento"), "yyyyMM")
                    != F.col("periodo_fuente")
                ),
                F.lit("MES_DISTINTO_DE_PERIODO_FUENTE")
            ),
            F.when(
                (F.col("con_tarjeta") == 1)
                & (
                    F.col("ordinal_de_tramo").isNull()
                    | (F.col("ordinal_de_tramo") <= 0)
                ),
                F.lit("ORDINAL_INVALIDO_CON_TARJETA")
            ),
            F.when(
                (F.col("con_tarjeta") == 0)
                & (F.col("ordinal_de_tramo") < 0),
                F.lit("ORDINAL_NEGATIVO_SIN_TARJETA")
            )
        ),
        lambda valor: valor.isNotNull()
    )
)

In [ ]:
# Resumen de validaciones: total de filas y cantidad por regla
reglas_validacion = [
    "CON_TARJETA_INVALIDA_O_AUSENTE",
    "CANTIDAD_PASAJEROS_INVALIDA",
    "FECHA_EVENTO_AUSENTE",
    "PERIODO_FUENTE_INVALIDO",
    "MES_DISTINTO_DE_PERIODO_FUENTE",
    "ORDINAL_INVALIDO_CON_TARJETA",
    "ORDINAL_NEGATIVO_SIN_TARJETA",
]

resumen_validacion = viajes_completados.agg(
    F.count("*").alias("total_filas"),
    *[
        F.sum(
            F.array_contains(
                F.col("problemas_validacion"), regla
            ).cast("int")
        ).alias(regla)
        for regla in reglas_validacion
    ]
)

display(resumen_validacion)

# Muestra opcional de filas con problemas
# display(
#     viajes_completados.filter(
#         F.size("problemas_validacion") > 0
#     ).limit(100)
# )

**Definicion de atributos para la tabla silver**

In [ ]:
viajes_final = viajes_completados.select(
    "id_viaje",
    "con_tarjeta",
    "fecha_evento",
    "tipo_viaje",
    "descripcion_tipo_viaje",
    "grupo_usuario",
    "descripcion_grupo_usuario",
    "grupo_usuario_especifico",
    "descripcion_grupo_usuario_espe",
    "ordinal_de_tramo",
    "cantidad_pasajeros",
    "codigo_parada_origen",
    "cod_empresa",
    "descrip_empresa",
    "linea_codigo",
    "dsc_linea",
    "sevar_codigo",
    "periodo_fuente",
    "problemas_validacion"
)


## **Horarios de Paradas**

Al igual que en la tabla anterior, se comienza analizando el esquema de la tabla y viendo una muestra de sus registros. 

`tipo_dia`:	Tipo de día del servicio: 1 hábil, 2 sábado, 3 domingo, según el diccionario.

`cod_variante`:	Código de la variante del recorrido.

`frecuencia`:	Identificación de la salida en formato HMM0; no representa minutos entre servicios.

`cod_ubic_parada`:	Código de la parada.

`ordinal`:	Posición de la parada dentro del recorrido.

`hora`:	Hora teórica de pasada en formato HMM.

`dia_anterior`:	Relación entre el día de inicio y el de pasada: N mismo día; S inicio el día anterior; * refiere a un caso especial.

`archivo_origen`:	Nombre del CSV cargado.

In [ ]:
paradas = spark.table(
    f"{CATALOG}.{BRONZE}."
    "horarios_por_parada_uptu_pasada_variante"
)

paradas.printSchema()
display(paradas.limit(10))

**Resumen de faltantes**

In [ ]:
columnas_paradas = paradas.columns

paradas_revisadas = paradas.withColumn(
    "columnas_faltantes",
    F.filter(
        F.array(*[
            F.when(
                F.col(c).isNull()
                | (F.trim(F.col(c).cast("string")) == ""),
                F.lit(c)
            )
            for c in columnas_paradas
        ]),
        lambda elemento: elemento.isNotNull()
    )
)

display(
    paradas_revisadas.agg(
        F.count("*").alias("total_registros"),
        *[
            F.count(
                F.when(
                    F.array_contains(
                        F.col("columnas_faltantes"), c
                    ),
                    F.lit(1)
                )
            ).alias(c)
            for c in columnas_paradas
        ]
    )
)

El resumen de faltantes se muestra arriba. Los resultados dependen de la ejecución de las celdas anteriores.

## Limpieza de espacios y normalización

Antes de convertir tipos, se quitan los espacios externos, se convierten las cadenas vacías a NULL y se normaliza `dia_anterior` a mayúsculas. Se mantiene `archivo_origen`.

In [ ]:
paradas_limpias = paradas.select(*[
    F.when(
        F.trim(F.col(c)) == "",
        F.lit(None).cast("string")
    )
    .otherwise(F.trim(F.col(c)))
    .alias(c)
    for c in columnas_paradas
])

paradas_limpias = paradas_limpias.withColumn(
    "dia_anterior",
    F.upper(F.col("dia_anterior"))
)

# Visualización opcional de datos después de trim
# display(paradas_limpias.limit(10))

**Convertir columnas numéricas**

In [ ]:
tipos_paradas = {
    "tipo_dia": "INT",
    "cod_variante": "INT",
    "frecuencia": "INT",
    "cod_ubic_parada": "INT",
    "ordinal": "INT",
    "hora": "INT",
}

paradas_tipadas = paradas_limpias

for columna, tipo in tipos_paradas.items():
    paradas_tipadas = paradas_tipadas.withColumn(
        f"{columna}_tipado",
        F.expr(f"try_cast(`{columna}` AS {tipo})")
    )

paradas_tipadas = paradas_tipadas.withColumn(
    "errores_conversion",
    F.filter(
        F.array(*[
            F.when(
                F.col(c).isNotNull()
                & F.col(f"{c}_tipado").isNull(),
                F.lit(c)
            )
            for c in tipos_paradas
        ]),
        lambda valor: valor.isNotNull()
    )
)

# Visualización opcional de errores de conversión
# display(
#     paradas_tipadas.filter(F.size("errores_conversion") > 0)
# )

**Formalizar la hora y el tipo de día**

La hora original ya viene codificada como HMM; se interpreta, no se transforma originalmente a HMM.

se conserva `dia_anterior`, ya que la hora por sí sola no indica a qué día corresponde la pasada.

La hora publicada está codificada como HMM. Se separan sus componentes para validar horas y minutos, y se obtiene una representación en segundos desde medianoche que permite realizar cálculos temporales. Útil para rescatar metricas agrupando por horas concretas.

Además, se agrega una columna `tipo_dia_normalizado` que identifica la categoría del servicio programado, no un viaje efectivamente realizado.


In [ ]:
paradas_tipadas = (
    paradas_tipadas
    .withColumn(
        "hora_h",
        F.floor(F.col("hora_tipado") / 100).cast("int")
    )
    .withColumn(
        "hora_m",
        (F.col("hora_tipado") % 100).cast("int")
    )
    .withColumn(
        "tipo_dia_normalizado",
        F.when(F.col("tipo_dia_tipado") == 1, "HABIL")
         .when(F.col("tipo_dia_tipado") == 2, "SABADO")
         .when(F.col("tipo_dia_tipado") == 3, "DOMINGO")
    )
)

hora_valida_paradas = (
    F.col("hora_h").between(0, 23)
    & F.col("hora_m").between(0, 59)
)

paradas_tipadas = paradas_tipadas.withColumn(
    "segundos_desde_medianoche",
    F.when(
        hora_valida_paradas,
        F.col("hora_h") * 3600 + F.col("hora_m") * 60
    )
)

# Visualización opcional de descomposición de horas
# display(
#     paradas_tipadas.select(
#         "hora",
#         "hora_h",
#         "hora_m",
#         "segundos_desde_medianoche",
#         "tipo_dia_normalizado",
#         "dia_anterior"
#     ).limit(10)
# )

**Mostrar valores que requieren revisión**

Los problemas se marcan para revisión. No afirmar que están aislados si no existe una salida separada.

In [ ]:
paradas_tipadas = paradas_tipadas.withColumn(
    "problemas_validacion",
    F.filter(
        F.array(
            F.when(
                F.col("tipo_dia_normalizado").isNull(),
                F.lit("TIPO_DIA_INVALIDO_O_AUSENTE")
            ),
            F.when(
                F.col("hora_tipado").isNull()
                | ~hora_valida_paradas,
                F.lit("HORA_PARA_REVISAR")
            ),
            F.when(
                F.col("ordinal_tipado").isNull()
                | (F.col("ordinal_tipado") <= 0),
                F.lit("ORDINAL_PARA_REVISAR")
            ),
            F.when(
                F.col("dia_anterior").isNull()
                | ~F.col("dia_anterior").isin("N", "S", "*"),
                F.lit("DIA_ANTERIOR_INVALIDO_O_AUSENTE")
            )
        ),
        lambda valor: valor.isNotNull()
    )
)

# Visualización opcional de filas con problemas
# display(
#     paradas_tipadas.filter(
#         F.size("problemas_validacion") > 0
#     )
# )

El resumen de validación se muestra arriba. Los resultados dependen de la ejecución de las celdas anteriores.

**Revisar posibles duplicados**

In [ ]:
claves_paradas = [
    "tipo_dia_tipado",
    "cod_variante_tipado",
    "frecuencia_tipado",
    "ordinal_tipado",
    "dia_anterior",
]

duplicados_paradas = (
    paradas_tipadas
    .groupBy(*claves_paradas)
    .count()
    .filter(F.col("count") > 1)
)

display(duplicados_paradas)

## `paradas_final`

Se seleccionan las columnas originales sustituyendo las numéricas por sus columnas `_tipado` con el nombre original. Se conservan `tipo_dia_normalizado`, `segundos_desde_medianoche`, `errores_conversion` y `problemas_validacion`. Las auxiliares `hora_h` y `hora_m` no se incluyen. No se escribe todavía la tabla.

In [ ]:
paradas_final = paradas_tipadas.select(
    *[
        F.col(f"{c}_tipado").alias(c)
        if c in tipos_paradas
        else F.col(c)
        for c in columnas_paradas
    ],
    "tipo_dia_normalizado",
    "segundos_desde_medianoche",
    "errores_conversion",
    "problemas_validacion",
)

paradas_final.printSchema()
# display(paradas_final.limit(10))

## **Horarios de Paradas Circulares**


Se detallan las variables únicas del dataset

`cod_circular`:	Código de la variante circular.

`cod_variante`:	Código de la variante que realiza esa parte del circular.

`parte`:	Identifica la primera o segunda parte del circular.

`ordinal`: Posición de la parada dentro de la variante que integra el circular.

**Priemeras lecturas y visualización**

In [ ]:
circulares = spark.table(
    f"{CATALOG}.{BRONZE}."
    "horarios_por_parada_uptu_pasada_circular"
)

circulares.printSchema()
display(circulares.limit(10))

A simple vista, podemos detectar que la mayor diferencia de este con el resto de datasets, es que presenta un rellenado a la izquierda en todos sus campos, más adelantes se eliminarán para facilitar un posterior análisis, relaciones con otras tablas y operación con los datos.

**Mostrar registros con nulos o vacíos**

In [ ]:
columnas_circulares = circulares.columns

circulares = circulares.withColumn(
    "columnas_faltantes",
    F.filter(
        F.array(*[
            F.when(
                F.col(c).isNull()
                | (F.trim(F.col(c).cast("string")) == ""),
                F.lit(c)
            )
            for c in columnas_circulares
        ]),
        lambda valor: valor.isNotNull()
    )
)

# Visualización opcional de registros con faltantes
# display(
#     circulares.filter(F.size("columnas_faltantes") > 0)
# )

**Resumen de faltantes**

In [ ]:
display(
    circulares.agg(
        F.count("*").alias("total_registros"),
        *[
            F.count(
                F.when(
                    F.array_contains(
                        F.col("columnas_faltantes"), c
                    ),
                    F.lit(1)
                )
            ).alias(c)
            for c in columnas_circulares
        ]
    )
)

**Quitar espacios y unificar vacíos**

In [ ]:
circulares_limpias = circulares.select(*[
    F.when(
        F.trim(F.col(c)) == "",
        F.lit(None).cast("string")
    )
    .otherwise(F.trim(F.col(c)))
    .alias(c)
    for c in columnas_circulares
])

circulares_limpias = circulares_limpias.withColumn(
    "dia_anterior",
    F.upper(F.col("dia_anterior"))
)

# Visualización opcional de datos después de trim
# display(circulares_limpias.limit(10))

**Convertir números y eliminar el relleno de ceros**

La conversión numérica elimina los ceros a la izquierda presentes en este dataset: `000000591` se convierte en `591`.


In [ ]:
tipos_circulares = {
    "tipo_dia": "INT",
    "cod_circular": "INT",
    "frecuencia": "INT",
    "cod_ubic_parada": "INT",
    "cod_variante": "INT",
    "parte": "INT",
    "ordinal": "INT",
    "hora": "INT",
}

circulares_tipadas = circulares_limpias

for columna, tipo in tipos_circulares.items():
    circulares_tipadas = circulares_tipadas.withColumn(
        f"{columna}_tipado",
        F.expr(f"try_cast(`{columna}` AS {tipo})")
    )

circulares_tipadas = circulares_tipadas.withColumn(
    "errores_conversion",
    F.filter(
        F.array(*[
            F.when(
                F.col(c).isNotNull()
                & F.col(f"{c}_tipado").isNull(),
                F.lit(c)
            )
            for c in tipos_circulares
        ]),
        lambda valor: valor.isNotNull()
    )
)

display(
    circulares_tipadas.filter(
        F.size("errores_conversion") > 0
    )
)

Comprobamos que se haya efectuado el cambio anterior

In [ ]:
# Visualización opcional de conversión numérica
# display(
#     circulares_tipadas.select(
#         "cod_variante",
#         "cod_variante_tipado",
#         "cod_ubic_parada",
#         "cod_ubic_parada_tipado",
#         "hora",
#         "hora_tipado"
#     ).limit(10)
# )

**Formalizar la hora y el tipo de día**

In [ ]:
circulares_tipadas = (
    circulares_tipadas
    .withColumn(
        "hora_h",
        F.floor(F.col("hora_tipado") / 100).cast("int")
    )
    .withColumn(
        "hora_m",
        (F.col("hora_tipado") % 100).cast("int")
    )
    .withColumn(
        "tipo_dia_normalizado",
        F.when(F.col("tipo_dia_tipado") == 1, "HABIL")
         .when(F.col("tipo_dia_tipado") == 2, "SABADO")
         .when(F.col("tipo_dia_tipado") == 3, "DOMINGO")
         .when(F.col("tipo_dia_tipado").isNotNull(), "NO_DOCUMENTADO")
         .otherwise("SIN_DATO_O_NO_INTERPRETABLE")
    )
)

hora_valida_circulares = (
    F.col("hora_h").between(0, 23)
    & F.col("hora_m").between(0, 59)
)

circulares_tipadas = circulares_tipadas.withColumn(
    "segundos_desde_medianoche",
    F.when(
        hora_valida_circulares,
        F.col("hora_h") * 3600 + F.col("hora_m") * 60
    )
)

# Visualización opcional de descomposición de horas
# display(
#     circulares_tipadas.select(
#         "hora",
#         "hora_h",
#         "hora_m",
#         "segundos_desde_medianoche",
#         "tipo_dia_normalizado",
#         "dia_anterior"
#     ).limit(10)
# )

**Mostrar valores que requieren revisión**


In [ ]:
circulares_tipadas = circulares_tipadas.withColumn(
    "problemas_validacion",
    F.filter(
        F.array(
            F.when(
                F.col("tipo_dia_normalizado") == "SIN_DATO_O_NO_INTERPRETABLE",
                F.lit("TIPO_DIA_AUSENTE_O_NO_INTERPRETABLE")
            ),
            F.when(
                F.col("tipo_dia_normalizado") == "NO_DOCUMENTADO",
                F.lit("TIPO_DIA_NO_DOCUMENTADO")
            ),
            F.when(
                F.col("hora_tipado").isNull()
                | ~hora_valida_circulares,
                F.lit("HORA_PARA_REVISAR")
            ),
            F.when(
                F.col("ordinal_tipado").isNull()
                | (F.col("ordinal_tipado") <= 0),
                F.lit("ORDINAL_PARA_REVISAR")
            ),
            F.when(
                F.col("parte_tipado").isNull()
                | ~F.col("parte_tipado").isin(1, 2),
                F.lit("PARTE_PARA_REVISAR")
            ),
            F.when(
                F.col("dia_anterior").isNull()
                | ~F.col("dia_anterior").isin("N", "S", "*"),
                F.lit("DIA_ANTERIOR_INVALIDO_O_AUSENTE")
            )
        ),
        lambda valor: valor.isNotNull()
    )
)

# Visualización opcional de filas con problemas
# display(
#     circulares_tipadas.filter(
#         F.size("problemas_validacion") > 0
#     )
# )

Los resultados dependen de la descarga cargada en bronze. La distribución por `tipo_dia_tipado` se calcula en la celda siguiente sobre esa tabla.

In [ ]:
# Distribución de registros por tipo_dia_tipado en la tabla cargada en bronze
distribucion_tipo_dia_circulares = (
    circulares_tipadas
    .groupBy("tipo_dia_tipado")
    .agg(F.count("*").alias("registros"))
    .orderBy("tipo_dia_tipado")
)

display(distribucion_tipo_dia_circulares)

**Revisar posibles duplicados**

In [ ]:
claves_circulares = [
    "tipo_dia_tipado",
    "cod_circular_tipado",
    "frecuencia_tipado",
    "parte_tipado",
    "cod_variante_tipado",
    "ordinal_tipado",
    "dia_anterior",
]

duplicados_circulares = (
    circulares_tipadas
    .groupBy(*claves_circulares)
    .count()
    .filter(F.col("count") > 1)
)

display(duplicados_circulares)

**Descarte de columnas con ceros agregados**

Una vez realizada la limpieza, nos quedamos unicamente con aquellas columnas pasadas por el proceso de curado y agregadas para un posterior análisis.

In [ ]:
circulares_final = circulares_tipadas.select(
    *[
        F.col(f"{c}_tipado").alias(c)
        if c in tipos_circulares
        else F.col(c)
        for c in columnas_circulares
    ],
    "tipo_dia_normalizado",
    "segundos_desde_medianoche",
    "errores_conversion",
    "problemas_validacion",
)

circulares_final.printSchema()
# display(circulares_final.limit(10))

## **Horarios de Puntos de Control**

`cod_linea`:	Código interno de la línea.

`linea`:	Nombre público de la línea.

`cod_sublinea`:	Código de la sublínea.

`sublinea`:	Descripción de la sublínea.

`variante`:	Código de la variante del recorrido.

`codigo_minuta`:	Clasificación del horario: 1/2 hábil, 3/4 sábado, 5/6 domingo.

`nro_frecuencia`:	Identificación de la salida del recorrido.

`codigo_punto`:	Código del punto de control.

`hora`:	Hora teórica en el punto, codificada como HHMMSS.

`fecha_desde`:	Fecha desde la que está vigente el horario.

`archivo_origen`:	Nombre del CSV cargado.

**Leer y visualizar**

In [ ]:
control = spark.table(
    f"{CATALOG}.{BRONZE}."
    "horarios_puntos_control_horarios_omnibus_datos"
)

control.printSchema()
display(control.limit(10))

**Resumen de faltantes**

In [ ]:
columnas_control = control.columns

control = control.withColumn(
    "columnas_faltantes",
    F.filter(
        F.array(*[
            F.when(
                F.col(c).isNull()
                | (F.trim(F.col(c).cast("string")) == ""),
                F.lit(c)
            )
            for c in columnas_control
        ]),
        lambda valor: valor.isNotNull()
    )
)

display(
    control.agg(
        F.count("*").alias("total_registros"),
        *[
            F.count(
                F.when(
                    F.array_contains(
                        F.col("columnas_faltantes"), c
                    ),
                    F.lit(1)
                )
            ).alias(c)
            for c in columnas_control
        ]
    )
)

El resumen de faltantes se muestra arriba. Los resultados dependen de la ejecución de las celdas anteriores.

**Quitar espacios y unificar vacíos**

In [ ]:
control_limpio = control.select(*[
    F.when(
        F.trim(F.col(c)) == "",
        F.lit(None).cast("string")
    )
    .otherwise(F.trim(F.col(c)))
    .alias(c)
    for c in columnas_control
])

# Visualización opcional de datos después de trim
# display(control_limpio.limit(10))

**Identificar el encabezado repetido**

In [ ]:
control_limpio = control_limpio.withColumn(
    "es_encabezado_repetido",
    F.coalesce(
        (F.lower(F.col("cod_linea")) == "cod_linea")
        & (F.lower(F.col("variante")) == "variante")
        & (
            F.lower(F.col("codigo_minuta"))
            == "codigo_minuta"
        ),
        F.lit(False)
    )
)

# Visualización opcional del encabezado repetido
# display(
#     control_limpio.filter(F.col("es_encabezado_repetido"))
# )

**Eliminación del encabezado repetido**

Se identificó una fila que repite los nombres de las columnas dentro del contenido del CSV. Se separa antes de convertir tipos porque representa documentación estructural y no una pasada de ómnibus.

Una vez filtrada, la columna `es_encabezado_repetido` queda en `false` para todos los registros restantes. No es necesario volver a filtrar por ella en celdas posteriores.

In [ ]:
encabezados_control = control_limpio.filter(
    F.col("es_encabezado_repetido")
)

control_limpio = control_limpio.filter(
    ~F.col("es_encabezado_repetido")
)

**Convertir columnas numéricas**

`linea` y `sublinea` permanecen como texto: son el nombre público y la descripción.

In [ ]:
tipos_control = {
    "cod_linea": "BIGINT",
    "cod_sublinea": "BIGINT",
    "variante": "BIGINT",
    "codigo_minuta": "INT",
    "nro_frecuencia": "BIGINT",
    "codigo_punto": "BIGINT",
    "hora": "INT",
}

control_tipado = control_limpio

for columna, tipo in tipos_control.items():
    control_tipado = control_tipado.withColumn(
        f"{columna}_tipado",
        F.expr(f"try_cast(`{columna}` AS {tipo})")
    )

control_tipado = control_tipado.withColumn(
    "errores_conversion",
    F.filter(
        F.array(*[
            F.when(
                F.col(c).isNotNull()
                & F.col(f"{c}_tipado").isNull(),
                F.lit(c)
            )
            for c in tipos_control
        ]),
        lambda valor: valor.isNotNull()
    )
)

# Visualización opcional de errores de conversión
# display(
#     control_tipado.filter(
#         F.size("errores_conversion") > 0
#     )
# )

**Convertir `fecha_desde`**

`fecha_desde` indica desde qué fecha está vigente el horario; no es la fecha del evento. Se convierte a `DATE` para permitir joins y ordenamientos por vigencia.

In [ ]:
control_tipado = control_tipado.withColumn(
    "fecha_desde_tipado",
    F.coalesce(
        F.expr(
            "try_to_timestamp(fecha_desde, 'yyyy-MM-dd')"
        ),
        F.expr(
            "try_to_timestamp(upper(fecha_desde), 'dd-MMM-yy')"
        )
    ).cast("date")
)

# Visualización opcional de fechas no convertidas
# display(
#     control_tipado.filter(
#         F.col("fecha_desde_tipado").isNull()
#     ).select(
#         "cod_linea",
#         "variante",
#         "fecha_desde",
#         "fecha_desde_tipado"
#     )
# )

Acá la hora usa tambien los segundos (`HHMMSS`): `94400` se interpreta como `09:44:00`.

El día de servicio va de 04:00 a 04:00. `segundos_desde_medianoche` no resuelve por sí solo el día de servicio ni los cruces de medianoche.

In [ ]:
control_tipado = (
    control_tipado
    .withColumn(
        "hora_h",
        F.floor(F.col("hora_tipado") / 10000).cast("int")
    )
    .withColumn(
        "hora_m",
        (
            F.floor(F.col("hora_tipado") / 100) % 100
        ).cast("int")
    )
    .withColumn(
        "hora_s",
        (F.col("hora_tipado") % 100).cast("int")
    )
    .withColumn(
        "tipo_dia_normalizado",
        F.when(
            F.col("codigo_minuta_tipado").isin(1, 2),
            "HABIL"
        )
        .when(
            F.col("codigo_minuta_tipado").isin(3, 4),
            "SABADO"
        )
        .when(
            F.col("codigo_minuta_tipado").isin(5, 6),
            "DOMINGO"
        )
    )
)

hora_valida_control = (
    F.col("hora_h").between(0, 23)
    & F.col("hora_m").between(0, 59)
    & F.col("hora_s").between(0, 59)
)

control_tipado = control_tipado.withColumn(
    "segundos_desde_medianoche",
    F.when(
        hora_valida_control,
        F.col("hora_h") * 3600
        + F.col("hora_m") * 60
        + F.col("hora_s")
    )
)

# Visualización opcional de descomposición de horas
# display(
#     control_tipado.select(
#         "hora",
#         "hora_h",
#         "hora_m",
#         "hora_s",
#         "segundos_desde_medianoche",
#         "codigo_minuta",
#         "tipo_dia_normalizado"
#     ).limit(10)
# )

**Mostrar valores que requieren revisión**

In [ ]:
control_tipado = control_tipado.withColumn(
    "problemas_validacion",
    F.filter(
        F.array(
            F.when(
                F.col("fecha_desde_tipado").isNull(),
                F.lit("FECHA_DESDE_INVALIDA_O_AUSENTE")
            ),
            F.when(
                F.col("hora_tipado").isNull()
                | ~hora_valida_control,
                F.lit("HORA_PARA_REVISAR")
            ),
            F.when(
                F.col("tipo_dia_normalizado").isNull(),
                F.lit("MINUTA_INVALIDA_O_AUSENTE")
            )
        ),
        lambda valor: valor.isNotNull()
    )
)

# Visualización opcional de filas con problemas
# display(
#     control_tipado.filter(
#         F.size("problemas_validacion") > 0
#     )
# )

**Revisar posibles duplicados**

In [ ]:
claves_control = [
    "cod_linea_tipado",
    "cod_sublinea_tipado",
    "variante_tipado",
    "codigo_minuta_tipado",
    "nro_frecuencia_tipado",
    "codigo_punto_tipado",
    "fecha_desde_tipado",
]

duplicados_control = (
    control_tipado
    .groupBy(*claves_control)
    .count()
    .filter(F.col("count") > 1)
)

display(duplicados_control)

## **Control Final**

Se seleccionan las columnas originales sustituyendo las numéricas por sus columnas `_tipado` con el nombre original. `fecha_desde` se reemplaza por `fecha_desde_tipado`. Se conservan `tipo_dia_normalizado`, `segundos_desde_medianoche`, `errores_conversion` y `problemas_validacion`. Se excluyen `es_encabezado_repetido`, `hora_h`, `hora_m`, `hora_s` y `columnas_faltantes`. No se escribe todavía la tabla.

In [ ]:
control_final = control_tipado.select(
    *[
        F.col(f"{c}_tipado").alias(c)
        if c in tipos_control
        else F.col(c)
        for c in columnas_control
        if c != "fecha_desde"
    ],
    F.col("fecha_desde_tipado").alias("fecha_desde"),
    "tipo_dia_normalizado",
    "segundos_desde_medianoche",
    "errores_conversion",
    "problemas_validacion",
)

control_final.printSchema()
# display(control_final.limit(10))

## **Correspondencia entre tablas**

Poaterior a la limpieza, se identifica una posible correspondencia entre variables presentes entre tablas, a continuacion se plantean diferentes operaciones de join que las confirmen.

`sevar_codigo` de _stm_viajes_ con	`cod_variante` en _horarios por parada_ o _variante en puntos de control_.

`linea_codigo` y `cod_linea` en _puntos de control_.

`codigo_parada_origen` y `cod_ubic_parada` en _horarios por parada_.

## **Publicación de tablas**

**viajes**

In [ ]:
(
    viajes_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER}.viajes_stm")
)

print("Tabla publicada:", f"{CATALOG}.{SILVER}.viajes_stm")

**horarios por parada**

In [ ]:
(
    paradas_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER}.horarios_por_parada")
)

print("Tabla publicada:", f"{CATALOG}.{SILVER}.horarios_por_parada")

**horarios por parada circular**

In [ ]:
(
    circulares_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER}.horarios_circulares")
)

print("Tabla publicada:", f"{CATALOG}.{SILVER}.horarios_circulares")

**horarios de puntos de control**

In [ ]:
(
    control_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER}.horarios_puntos_control")
)

print(
    "Tabla publicada:",
    f"{CATALOG}.{SILVER}.horarios_puntos_control"
)

**Comprobacion de tablas persistidas**

In [ ]:
display(
    spark.sql(f"SHOW TABLES IN {CATALOG}.{SILVER}")
)

tablas_silver = [
    "viajes_stm",
    "horarios_por_parada",
    "horarios_circulares",
    "horarios_puntos_control"
]

for tabla in tablas_silver:
    nombre = f"{CATALOG}.{SILVER}.{tabla}"

    print(f"\nTabla: {nombre}")

    guardada = spark.table(nombre)
    guardada.printSchema()
    display(guardada.limit(10))